In [1]:
import pandas as pd
import os
import gc

In [3]:
folder_path = '/home/cloudcraftz/Documents/BN_WK_2023_ALL/2023_2/'

for filename in os.listdir(folder_path):
    if os.path.isfile(os.path.join(folder_path, filename)):
        if filename.endswith('.csv'):
            absolute_path = os.path.abspath(os.path.join(folder_path, filename))
            # print(absolute_path)
 
            #!Rename files
            # parts = filename.split("_Intraday_Preprocessed")
            # # print(parts)
            # df = pd.read_csv(absolute_path)
            # expiry_date = df['ExpiryDate'].unique()
            # date = expiry_date[0].replace("-", "")
            # new_filename = f"{parts[0]}_"+date+"_Intraday_Preprocessed"+parts[1]
            # print(new_filename)
            
            # # # # Construct the full paths for the old and new filenames
            # old_file = os.path.join(folder_path, filename)
            # new_file = os.path.join(folder_path, new_filename)
            
            # # # # Rename the file
            # os.rename(old_file, new_file)
            # print(f"Renamed: {old_file} -> {new_file}")
 
            
            
            
            #!Spot Create
            df = pd.read_csv(absolute_path)
            df = df[['Date Time', 'Spot']]
            df.drop_duplicates(inplace=True, ignore_index=True)
            date_str = filename.split('_')[1]
            print(date_str)
            # print(df.head())
            df.to_csv("/home/cloudcraftz/Documents/BN_WK_2023_ALL/2023_2/Spot/BANKNIFTY_"+date_str+"_Intraday_Spot.csv", index=False)
 
            del df
            gc.collect()

20231219
20230613
20230112
20230515
20231214
20231222
20230308
20231018
20231106
20230627
20230524
20230705
20230502
20230814
20230222
20230719
20230928
20231201
20231009
20230227
20230831
20230410
20230904
20230118
20230125
20230804
20230516
20230427
20230703
20230602
20230425
20231123
20230801
20230323
20230731
20230911
20231129
20231206
20230522
20231204
20230127
20230503
20230216
20230907
20231115
20230728
20230915
20230110
20230207
20230811
20230529
20230510
20231030
20231121
20230927
20231005
20231110
20230322
20230807
20230918
20230517
20230926
20230420
20230117
20230310
20230403
20231117
20231026
20230406
20230620
20230220
20231212
20230511
20230721
20230711
20231010
20231101
20230512
20231221
20230428
20230203
20230316
20230119
20230921
20230609
20230615
20231226
20230829
20230614
20230816
20231013
20231016
20230606
20230802
20231227
20230206
20230505
20230329
20230707
20230630
20230605
20231109
20230526
20230929
20230824
20231031
20230324
20230724
20230104
20230621
20231218
2

In [5]:
count

242

In [4]:
import glob
from datetime import datetime, timedelta
import os
import pandas as pd
import numpy as np
import json

def generate_trade_expiry_info(path, curr_year):
    try:
        file_dir = glob.glob(path+"/*.csv")
        trade_date_list = []
        expiry_date_list = []
        for file in file_dir:
            # print(file)
            name_list = file.split(os.path.sep)[-1].split("_")

            df = pd.read_csv(file)

            trade_date_list.append(datetime.strptime(name_list[1], "%Y%m%d").date())
            expiry_date_list.extend(list(map(lambda x: datetime.strptime(x, "%Y-%m-%d").date(), df['ExpiryDate'].unique())))

        trade_date_set = list(np.sort(list(set(trade_date_list))))
        expiry_date_set = list(np.sort(list(set(expiry_date_list))))
 
        expiry_date_set2 = list(map(lambda x: x.strftime("%Y-%m-%d"), expiry_date_set))
        trade_date_set2 = list(map(lambda x: x.strftime("%Y-%m-%d"), trade_date_set))
 
        unwind_dict = {}
        for date in expiry_date_set:
            if date in trade_date_set:
                unwind_dict[date.strftime("%Y-%m-%d")] = date.strftime("%Y-%m-%d")
        
            else:
                if date.year > curr_year:
                    last_trading_day_of_the_year = trade_date_set[-1]
        
                    print(f"For {date} unwind date is {last_trading_day_of_the_year}")
                    unwind_dict[date.strftime("%Y-%m-%d")] = last_trading_day_of_the_year.strftime("%Y-%m-%d")
        
                else:
                    is_run = True
                    date1 = date
                    while is_run:
                        date1 = date1 - timedelta(days=1)
        
                        if date1 in trade_date_set:
                            print(f"For {date} unwind date is {date1}")
                            unwind_dict[date.strftime("%Y-%m-%d")] = date1.strftime("%Y-%m-%d")
                            is_run = False
 
        data_info = {
            "Trading_dates": trade_date_set2,
            "Expiry_dates": expiry_date_set2,
            "Unwind_dates": unwind_dict,
        }
 
 
        ## Function to handle serialization of date objects
        def convert_to_json_serializable(obj):
            if isinstance(obj, date):
                return obj.isoformat()  # Convert date to string
            raise TypeError(f"Object of type {type(obj)} is not JSON serializable")
 
 
        json_file_path = os.path.join(path, "meta_data.json")
        # Save the dictionary to a JSON file
        print(json_file_path)
        with open(json_file_path, 'w') as json_file:
            json.dump(data_info, json_file, default=convert_to_json_serializable)
 
        print(f'Successfully saved the metadata')
 
    except Exception as ex:
        # logger.critical(f'error while generate_trade_expiry_info at line={get_exception_line_no()}. # {ex}')
        print(ex)

generate_trade_expiry_info(path="/home/cloudcraftz/Documents/BN_WK_2023_ALL/2023_2/", curr_year=2023)

For 2023-06-29 unwind date is 2023-06-27
/home/cloudcraftz/Documents/BN_WK_2023_ALL/2023_2/meta_data.json
Successfully saved the metadata
